In [ ]:
import subprocess, sys, os
import torch

# ═══════════════════════════════════════════════════════════════
# RDT2 TRAINING - Optimized for RTX 4090 24GB
# ═══════════════════════════════════════════════════════════════

# ── Configuration ───────────────────────────────────────────────
RDT2_DIR           = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
TASK               = "m750-pick-place"
DATASET_CONFIG     = "configs/datasets/custom_dataset.yaml"
OUTPUT_DIR         = f"/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvla-sft-{TASK}"
LOG_PATH           = f"/home/rishabh/Downloads/umi-pipeline-training/train_{TASK}.log"

# Model IDs (Official)
TOKENIZER_ID       = "Qwen/Qwen2.5-VL-7B-Instruct"
VAE_ID             = "robotics-diffusion-transformer/RVQActionTokenizer"
MODEL_ID           = "robotics-diffusion-transformer/RDT2-VQ"

# Training hyperparameters (OPTIMIZED FOR RTX 4090 24GB)
MAX_TRAIN_STEPS    = 3000
TRAIN_BATCH_SIZE   = 16         # ← Reduced from 64 to fit in 24GB
GRAD_ACCUM         = 4          # ← Increased to maintain effective batch=64
EVAL_BATCH_SIZE    = 16         # ← Reduced from 32
LOGGING_STEPS      = 25
CHECKPOINT_STEP    = 500
CHECKPOINT_LIMIT   = 6

# Optimizer settings (Official)
LR_SCHEDULER       = "cosine"
LEARNING_RATE      = 1e-5       # Official default
LR_WARMUP_STEPS    = 500        # Official default
MIXED_PRECISION    = "bf16"
DATALOADER_WORKERS = 8          # ← Reduced from 16 (half for stability)

# LoRA settings
USE_QLORA          = True
LORA_R             = 64
LORA_ALPHA         = 64
LORA_DROPOUT       = 0.1

# ── Environment Setup ───────────────────────────────────────────
os.chdir(RDT2_DIR)

os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION']  = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']           = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']            = 'false'
os.environ['CUDA_LAUNCH_BLOCKING']              = '0'  # Don't block CUDA

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GPU Check ───────────────────────────────────────────────────
if not torch.cuda.is_available():
    print("❌ No GPU available!")
    exit(1)

# Clear GPU cache before starting
torch.cuda.empty_cache()

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_free = torch.cuda.mem_get_info(0)[0] / 1e9

print(f"✅ GPU: {gpu_name}")
print(f"   Total VRAM: {gpu_mem:.1f}GB")
print(f"   Free VRAM:  {gpu_free:.1f}GB")

# ── Training Command ────────────────────────────────────────────
cmd = [
    sys.executable, 'main.py',
    
    # Model configuration
    '--tokenizer_name',               TOKENIZER_ID,
    '--vae_name',                      VAE_ID,
    '--pretrained_model_name_or_path', MODEL_ID,
    '--output_dir',                    OUTPUT_DIR,
    
    # Training configuration (OPTIMIZED FOR 24GB GPU)
    '--train_batch_size',              str(TRAIN_BATCH_SIZE),      # 16
    '--gradient_accumulation_steps',   str(GRAD_ACCUM),            # 4
    '--eval_batch_size',               str(EVAL_BATCH_SIZE),       # 16
    '--max_train_steps',               str(MAX_TRAIN_STEPS),
    '--eval_strategy',                 'no',
    
    # Logging & checkpointing
    '--logging_steps',                 str(LOGGING_STEPS),
    '--checkpoints_total_limit',       str(CHECKPOINT_LIMIT),
    '--checkpointing_step',            str(CHECKPOINT_STEP),
    
    # Optimizer & scheduler (Official)
    '--lr_scheduler',                  LR_SCHEDULER,
    '--learning_rate',                 str(LEARNING_RATE),
    '--lr_warmup_steps',               str(LR_WARMUP_STEPS),
    '--mixed_precision',               MIXED_PRECISION,
    
    # Performance
    '--dataloader_num_workers',        str(DATALOADER_WORKERS),
    '--gradient_checkpointing',
    
    # Logging & dataset
    '--log_level',                     'info',
    '--report_to',                     'tensorboard',
    '--dataset',                       DATASET_CONFIG,
    '--image_corruption',
    '--use_default_collate_fn_for_eval',
    
    # LoRA
    '--use_qlora',
    '--lora_r',                        str(LORA_R),
    '--lora_alpha',                    str(LORA_ALPHA),
    '--lora_dropout',                  str(LORA_DROPOUT),
]

# ── Training Summary ────────────────────────────────────────────
print("\n" + "="*70)
print("🚀 RDT2-VQ TRAINING - Optimized for RTX 4090 24GB")
print("="*70)
print(f"Task:            {TASK}")
print(f"Output:          {OUTPUT_DIR}")
print(f"Dataset:         {DATASET_CONFIG}")
print("\nTraining Configuration:")
print(f"  Total steps:     {MAX_TRAIN_STEPS}")
print(f"  Batch size:      {TRAIN_BATCH_SIZE} (adjusted for 24GB)")
print(f"  Grad accum:      {GRAD_ACCUM} (to maintain effective batch)")
print(f"  Effective batch: {TRAIN_BATCH_SIZE * GRAD_ACCUM} = 64 ✓ (MATCHES OFFICIAL)")
print(f"  Eval batch:      {EVAL_BATCH_SIZE}")
print(f"  Learning rate:   {LEARNING_RATE} (OFFICIAL: 1e-5) ✓")
print(f"  LR scheduler:    {LR_SCHEDULER} (OFFICIAL: cosine) ✓")
print(f"  Warmup steps:    {LR_WARMUP_STEPS} (OFFICIAL: 500) ✓")
print(f"  Mixed precision: {MIXED_PRECISION} (OFFICIAL: bf16) ✓")
print(f"  Num workers:     {DATALOADER_WORKERS}")
print(f"\nCheckpointing:")
print(f"  Save every:      {CHECKPOINT_STEP} steps")
print(f"  Keep last:       {CHECKPOINT_LIMIT} checkpoints")
print(f"\nLoRA Configuration:")
print(f"  Rank:            {LORA_R}")
print(f"  Alpha:           {LORA_ALPHA}")
print(f"  Dropout:         {LORA_DROPOUT}")
print(f"\nData Augmentation:")
print(f"  Image corruption: ✅")
print(f"\nLog file:          {LOG_PATH}")
print("="*70)

print(f"\n💾 Memory Optimization:")
print(f"  Batch 16 × Accum 4 = Effective 64 (same as official)")
print(f"  QLoRA rank 64: ~15-20GB VRAM")
print(f"  Gradient checkpointing: Enabled")
print(f"  Expected VRAM usage: ~18-22GB")
print(f"  Your available: {gpu_free:.1f}GB")
print(f"  ✅ Should fit comfortably!")

# Estimate training time
# With gradient accumulation, each step takes longer
seconds_per_step = 13.2 * GRAD_ACCUM  # Accumulation slows down
estimated_hours = (MAX_TRAIN_STEPS * seconds_per_step) / 3600

print(f"\n⏱️  Estimated training time:")
print(f"  ~{estimated_hours:.1f} hours")
print(f"  Note: Gradient accumulation adds overhead")
print("="*70)

print("\n📊 This Configuration:")
print("  ✅ Effective batch=64 (MATCHES official)")
print("  ✅ Learning rate=1e-5 (MATCHES official)")
print("  ✅ LR warmup=500 (MATCHES official)")
print("  ✅ Cosine schedule (MATCHES official)")
print("  ✅ Image corruption (MATCHES official)")
print("  ✅ Fits in RTX 4090 24GB")
print("="*70)

confirm = input(f"\n⚠️  Start training (~{estimated_hours:.1f}h)? (yes/no): ")
if confirm.lower() != 'yes':
    print("❌ Training cancelled.")
    exit(0)

# Clear cache one more time before training
torch.cuda.empty_cache()

# ── Run Training ────────────────────────────────────────────────
print("\n🏃 Training started...")
print(f"📝 Logging to: {LOG_PATH}\n")

with open(LOG_PATH, 'w') as logf:
    # Write header
    logf.write("="*70 + "\n")
    logf.write(f"RDT2-VQ Fine-tuning - RTX 4090 Optimized\n")
    logf.write("="*70 + "\n")
    logf.write(f"Task:              {TASK}\n")
    logf.write(f"Model:             {MODEL_ID}\n")
    logf.write(f"Dataset:           {DATASET_CONFIG}\n")
    logf.write(f"Output:            {OUTPUT_DIR}\n")
    logf.write(f"GPU:               {gpu_name} ({gpu_mem:.1f}GB)\n")
    logf.write(f"\nConfiguration:\n")
    logf.write(f"  train_batch_size:  {TRAIN_BATCH_SIZE}\n")
    logf.write(f"  grad_accum:        {GRAD_ACCUM}\n")
    logf.write(f"  effective_batch:   {TRAIN_BATCH_SIZE * GRAD_ACCUM} (= OFFICIAL 64) ✓\n")
    logf.write(f"  learning_rate:     {LEARNING_RATE}\n")
    logf.write(f"  lr_warmup_steps:   {LR_WARMUP_STEPS}\n")
    logf.write(f"  lr_scheduler:      {LR_SCHEDULER}\n")
    logf.write(f"  lora_rank:         {LORA_R}\n")
    logf.write(f"\nTotal steps:       {MAX_TRAIN_STEPS}\n")
    logf.write(f"Start time:        {__import__('datetime').datetime.now()}\n")
    logf.write("="*70 + "\n\n")
    logf.flush()
    
    # Run training
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy()
    )
    
    # Stream output
    for line in process.stdout:
        logf.write(line)
        logf.flush()
        print(line, end='', flush=True)

process.wait()

# ── Training Complete ───────────────────────────────────────────
if process.returncode == 0:
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("="*70)
    print(f"Output: {OUTPUT_DIR}")
    print(f"\nCheckpoints saved:")
    print(f"  📁 checkpoint-500")
    print(f"  📁 checkpoint-1000")
    print(f"  📁 checkpoint-1500  ← Likely best")
    print(f"  📁 checkpoint-2000  ← Also test this")
    print(f"  📁 checkpoint-2500")
    print(f"  📁 checkpoint-3000")
    print(f"\nLog: {LOG_PATH}")
    print("\n🎯 Next Steps:")
    print("  1. Test checkpoint-1500 and 2000 on robot")
    print("  2. Compare action diversity vs checkpoint-5000")
    print("  3. Expected: 50-400mm position range (vs 6mm before)")
    print("="*70)
else:
    print(f"\n❌ Training failed: exit code {process.returncode}")
    print(f"Check log: {LOG_PATH}")

✅ GPU: NVIDIA GeForce RTX 4090
   Total VRAM: 25.3GB
   Free VRAM:  24.8GB

🚀 RDT2-VQ TRAINING - Optimized for RTX 4090 24GB
Task:            m750-pick-place
Output:          /home/rishabh/Downloads/umi-pipeline-training/outputs/vqvla-sft-m750-pick-place
Dataset:         configs/datasets/custom_dataset.yaml

Training Configuration:
  Total steps:     3000
  Batch size:      16 (adjusted for 24GB)
  Grad accum:      4 (to maintain effective batch)
  Effective batch: 64 = 64 ✓ (MATCHES OFFICIAL)
  Eval batch:      16
  Learning rate:   1e-05 (OFFICIAL: 1e-5) ✓
  LR scheduler:    cosine (OFFICIAL: cosine) ✓
  Warmup steps:    500 (OFFICIAL: 500) ✓
  Mixed precision: bf16 (OFFICIAL: bf16) ✓
  Num workers:     8

Checkpointing:
  Save every:      500 steps
  Keep last:       6 checkpoints

LoRA Configuration:
  Rank:            64
  Alpha:           64
  Dropout:         0.1

Data Augmentation:
  Image corruption: ✅

Log file:          /home/rishabh/Downloads/umi-pipeline-training/train_m75

: 